# 02 — Tạo phiên, nhóm timestamp và chia tập theo thời gian

Notebook thực hiện bước F–G từ đầu ra notebook 01. Chạy các cell từ trên xuống bằng **Shift+Enter**, hoặc **Restart Kernel and Run All**; chọn kernel Python của môi trường ảo `.venv` trong project (đã `pip install -r requirements.txt`). Cell đầu in phiên bản Python/pandas/pyarrow thực tế.

Luồng xử lý: **events sạch → phiên → nhóm timestamp → mốc 80%/90% → loại phiên qua biên khỏi các tập sử dụng → lưu và kiểm tra lại**.

- Đầu vào: `data/processed/events_clean.parquet`, được giữ nguyên.
- Mở phiên mới khi đổi visitor hoặc khoảng nghỉ **lớn hơn 30 phút**. Đúng 30 phút vẫn cùng phiên.
- Nhóm sự kiện theo `(session_id, timestamp)`, giữ mọi item và loại hành vi trong nhóm.
- Chia thời gian theo toàn phiên; `boundary_drop` được lưu riêng để đối chiếu, không dùng làm lịch sử hoặc nhãn.
- Notebook độc lập với các file `.py` của project. Chạy lại sẽ cập nhật các đầu ra của notebook này.

Chưa tạo cặp prefix–target thực tế, tra metadata, mã hóa item theo train hoặc huấn luyện. Số mẫu trong báo cáo là số tính từ nhóm, trước lọc ứng viên.

## 1. Đường dẫn và cấu hình

Ngưỡng 30 phút tính bằng mili-giây. Mốc chia dùng phân vị timestamp của **sự kiện đã làm sạch**, không phải 80%/90% độ dài khoảng ngày. Tất cả thời gian giữ UTC.

In [ ]:
from pathlib import Path
import hashlib
import json
import sys
import pandas as pd
import pyarrow as pa
from IPython.display import display

cwd = Path.cwd().resolve()
ROOT = next(
    (path for path in [cwd, *cwd.parents]
     if (path / "data/processed/events_clean.parquet").is_file()), None
)
if ROOT is None:
    raise FileNotFoundError("Chạy notebook 01 trước, rồi mở notebook 02 từ project hoặc notebooks/.")

SOURCE = ROOT / "data/processed/events_clean.parquet"
PROCESSED = ROOT / "data/processed"
SPLIT_DIR = PROCESSED / "splits"
REPORT_DIR = ROOT / "reports/02_session_split"
UPSTREAM_REPORT = ROOT / "reports/01_events_clean/metrics.json"
INACTIVITY_MS = 30 * 60 * 1000
QUANTILES = [0.8, 0.9]
SPLIT_NAMES = ["train", "validation", "test", "boundary_drop"]

def require(condition, message):
    if not bool(condition):
        raise ValueError(message)


def fingerprint(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        while block := stream.read(8 * 1024 * 1024):
            digest.update(block)
    return {"bytes": path.stat().st_size, "sha256": digest.hexdigest()}


def content_sha256(frame, key):
    """SHA-256 theo nội dung bảng: sắp theo khóa chính rồi băm từng dòng.

    Không phụ thuộc cách pyarrow/pandas xếp byte trong tệp Parquet, nên dùng để
    đối chiếu giữa các máy. `fingerprint()` vẫn giữ mã băm theo byte để truy vết tệp.
    """
    ordered = frame.sort_values(key, kind="stable").reset_index(drop=True)
    digest = hashlib.sha256(json.dumps([str(c) for c in ordered.columns]).encode("utf-8"))
    row_hashes = pd.util.hash_pandas_object(ordered, index=False).to_numpy(dtype="<u8")
    digest.update(row_hashes.tobytes())
    return digest.hexdigest()


def write_json(path, value):
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + ".tmp")
    try:
        temporary.write_text(
            json.dumps(value, ensure_ascii=False, indent=2) + "\n",
            encoding="utf-8", newline="\n",
        )
        json.loads(temporary.read_text(encoding="utf-8"))
        temporary.replace(path)
    finally:
        temporary.unlink(missing_ok=True)


def utc(value):
    return pd.to_datetime(int(value), unit="ms", utc=True).isoformat()


print("Phiên bản:", sys.version.split()[0], "| pandas:", pd.__version__, "| pyarrow:", pa.__version__)
print("Project:", ROOT.name)
print("Đầu vào:", SOURCE.relative_to(ROOT).as_posix())
_content_example = pd.DataFrame({"event_id": [3, 1, 2], "value": ["c", "a", "b"]})
assert content_sha256(_content_example, ["event_id"]) == content_sha256(_content_example.iloc[::-1], ["event_id"])
assert content_sha256(_content_example, ["event_id"]) != content_sha256(_content_example.assign(value="x"), ["event_id"])
del _content_example
print("Ngưỡng phiên:", INACTIVITY_MS, "ms")

## 2. Đọc và kiểm tra đầu vào

Xác nhận đây là dữ liệu sạch: đủ cột, không thiếu trường bắt buộc, không trùng năm cột gốc, `event_id` duy nhất và `event_time` đúng timestamp UTC. Bước tạo phiên không xóa thêm sự kiện.

Nếu báo cáo notebook 01 đang có, mã băm của Parquet phải khớp báo cáo trước khi tiếp tục. Nếu không có báo cáo, vẫn kiểm tra trực tiếp dữ liệu; trạng thái nguồn trong báo cáo sẽ ghi rõ chưa đối chiếu nguồn trước đó.

In [ ]:
source_before = fingerprint(SOURCE)
events = pd.read_parquet(SOURCE, engine="pyarrow")
original_columns = ["timestamp", "visitorid", "event", "itemid", "transactionid"]
required_columns = original_columns + ["event_id", "event_time"]
missing_columns = set(required_columns) - set(events.columns)
require(not missing_columns, f"Thiếu cột đầu vào: {missing_columns}")
events = events[required_columns].copy()
require(len(events) > 0, "Đầu vào không có sự kiện.")
mandatory = ["timestamp", "visitorid", "event", "itemid", "event_id", "event_time"]
require(not events[mandatory].isna().any().any(), "Đầu vào còn thiếu trường bắt buộc.")
for column in ["timestamp", "visitorid", "itemid", "event_id", "transactionid"]:
    require(pd.api.types.is_integer_dtype(events[column].dtype), f"Cột {column} phải có kiểu số nguyên.")
    require(events[column].dropna().ge(0).all(), f"Cột {column} có số âm.")
require(events["event_id"].is_unique, "event_id không duy nhất.")
require(not events.duplicated(original_columns).any(), "Đầu vào còn trùng năm cột gốc; chạy lại notebook 01.")
require(events["event"].isin(["view", "addtocart", "transaction"]).all(), "Có nhãn hành vi không hợp lệ.")
is_purchase = events["event"].eq("transaction")
require(events.loc[is_purchase, "transactionid"].notna().all(), "Lượt mua thiếu transactionid.")
require(events.loc[~is_purchase, "transactionid"].isna().all(), "Hành vi không phải mua có transactionid; xem lại nguồn.")
require(isinstance(events["event_time"].dtype, pd.DatetimeTZDtype), "event_time phải là datetime có timezone.")
require(str(events["event_time"].dt.tz) == "UTC", "event_time chưa ở UTC.")
expected_time = pd.to_datetime(events["timestamp"], unit="ms", utc=True, errors="coerce")
require(expected_time.notna().all() and events["event_time"].eq(expected_time).all(), "event_time không khớp timestamp mili-giây.")
del expected_time

upstream_report = None
upstream_verified = False
if UPSTREAM_REPORT.is_file():
    upstream_report = json.loads(UPSTREAM_REPORT.read_text(encoding="utf-8"))
    require(
        upstream_report.get("output", {}).get("sha256") == source_before["sha256"],
        "Parquet khác báo cáo notebook 01. Chạy lại notebook 01 hoặc dùng đúng bộ dữ liệu/báo cáo tương ứng."
    )
    require(len(events) == upstream_report["clean_rows"], "Số dòng không khớp báo cáo notebook 01.")
    upstream_verified = True
    print("Đầu vào khớp SHA-256 và số dòng từ notebook 01.")
else:
    print("Không có báo cáo notebook 01; đầu vào được kiểm tra trực tiếp, chưa đối chiếu nguồn trước đó.")

input_rows = len(events)
events = events.sort_values(["visitorid", "timestamp", "event_id"]).reset_index(drop=True)
print(f"Sự kiện đầu vào: {input_rows:,}")
display(events.head())

## 3. Tạo phiên theo visitor và khoảng nghỉ

`shift()` lấy visitor/thời gian của dòng trước. Chỉ cần đổi visitor hoặc nghỉ hơn 1.800.000 ms là mở phiên mới. `cumsum()` đánh số phiên từ 1.

Ví dụ cùng visitor: 09:00 → 09:30 vẫn cùng phiên; 09:30 → 10:00:00.001 mở phiên mới. Sự kiện đồng thời có khoảng cách 0, vẫn cùng phiên.

Cell dưới kiểm tra ví dụ nhỏ cho: hai sự kiện đồng thời, đúng 30 phút, vượt 30 phút một mili-giây và đổi visitor. Ví dụ dùng cùng hàm tạo phiên như dữ liệu thật.

In [ ]:
def make_session_ids(frame, inactivity_ms):
    # frame phải được sắp theo visitor, timestamp và event_id.
    visitor_changes = frame["visitorid"].ne(frame["visitorid"].shift(fill_value=-1))
    gaps = frame.groupby("visitorid", sort=False)["timestamp"].diff()
    starts = (visitor_changes | gaps.gt(inactivity_ms)).fillna(True)
    return starts.cumsum().astype("int64")


example = pd.DataFrame({
    "visitorid": [1, 1, 1, 1, 2, 2],
    "timestamp": [0, 0, INACTIVITY_MS, 2 * INACTIVITY_MS + 1, 0, INACTIVITY_MS],
})
assert make_session_ids(example, INACTIVITY_MS).tolist() == [1, 1, 1, 2, 3, 3]
large_id_example = pd.DataFrame({
    "visitorid": [9223372036854775806, 9223372036854775806, 9223372036854775807],
    "timestamp": [0, 1, 0],
})
assert make_session_ids(large_id_example, INACTIVITY_MS).tolist() == [1, 1, 2]
del large_id_example
print("Đã kiểm tra ví dụ: đồng thời, đúng/vượt 30 phút và đổi visitor.")

events["session_id"] = make_session_ids(events, INACTIVITY_MS)
require(events.groupby("session_id", sort=False)["visitorid"].nunique().eq(1).all(), "Một phiên chứa nhiều visitor.")
internal_gaps = events.groupby("session_id", sort=False)["timestamp"].diff().dropna()
require(internal_gaps.between(0, INACTIVITY_MS).all(), "Khoảng nghỉ trong phiên sai ngưỡng hoặc thứ tự thời gian.")
print(f"Số phiên: {events['session_id'].nunique():,}")
display(events[["event_id", "visitorid", "timestamp", "event_time", "session_id"]].head(10))
del example, internal_gaps

## 4. Nhóm sự kiện cùng timestamp trong phiên

Mỗi nhóm có `group_id` duy nhất toàn bộ dữ liệu và `group_index` bắt đầu từ 1 trong từng phiên. Không tạo thứ tự giả giữa các sự kiện đồng thời.

Lưu hai bảng liên kết thay vì sao chép danh sách item/hành vi vào hàng triệu ô:

- Bảng `groups`: một dòng/nhóm, có thời gian và số sự kiện/item/hành vi.
- Bảng `events`: giữ đầy đủ từng sự kiện, bổ sung `group_id` và `group_index` để nối lại.

Do đó không mất thông tin xem/thêm giỏ/mua hay item lặp. Khi tạo target sau này, lấy tập item phân biệt của nhóm tiếp theo.

In [ ]:
new_group = (
    events["session_id"].ne(events["session_id"].shift(fill_value=-1))
    | events["timestamp"].ne(events["timestamp"].shift(fill_value=-1))
).fillna(True)
events["group_id"] = new_group.cumsum().astype("int64")

groups = events.loc[new_group, [
    "group_id", "session_id", "visitorid", "timestamp", "event_time"
]].copy().reset_index(drop=True)
groups["group_index"] = (groups.groupby("session_id", sort=False).cumcount() + 1).astype("int32")
group_stats = events.groupby("group_id", sort=False).agg(
    n_events=("event_id", "size"),
    n_items=("itemid", "nunique"),
    n_behaviors=("event", "nunique"),
)
groups = groups.join(group_stats, on="group_id", validate="one_to_one")
events["group_index"] = events["group_id"].map(groups.set_index("group_id")["group_index"]).astype("int32")

assert groups["group_id"].is_unique
assert not groups.duplicated(["session_id", "timestamp"]).any()
assert int(groups["n_events"].sum()) == input_rows
assert events[["group_id", "group_index"]].notna().all().all()
print(f"Nhóm timestamp: {len(groups):,}")
print(f"Nhóm có nhiều sự kiện: {int(groups['n_events'].gt(1).sum()):,}")
print(f"Nhóm có nhiều item: {int(groups['n_items'].gt(1).sum()):,}")
display(groups.head())

multiple_event_groups = groups.loc[groups["n_events"].gt(1)]
if not multiple_event_groups.empty:
    example_group_id = int(multiple_event_groups.iloc[0]["group_id"])
    print("Ví dụ các sự kiện đồng thời, group_id =", example_group_id)
    display(events.loc[events["group_id"].eq(example_group_id), [
        "session_id", "group_id", "group_index", "event_time", "itemid", "event"
    ]])
del new_group, group_stats, multiple_event_groups

## 5. Tóm tắt phiên và tính mốc chia thời gian

Mỗi phiên có thời gian đầu/cuối, số sự kiện và số nhóm. Phiên một nhóm không tạo được mẫu bước kế tiếp dù có thể chứa nhiều sự kiện đồng thời.

Phân vị dùng nội suy tuyến tính rồi lấy phần nguyên mili-giây, theo quy ước thống nhất của project. Tính mốc trên toàn bộ events sạch **trước** khi loại phiên qua biên và lưu cố định để các mô hình dùng chung.

In [ ]:
sessions = events.groupby("session_id", sort=False, as_index=False).agg(
    visitorid=("visitorid", "first"),
    start_ms=("timestamp", "min"),
    end_ms=("timestamp", "max"),
    n_events=("event_id", "size"),
    n_groups=("group_id", "nunique"),
)
sessions["start_time"] = pd.to_datetime(sessions["start_ms"], unit="ms", utc=True)
sessions["end_time"] = pd.to_datetime(sessions["end_ms"], unit="ms", utc=True)
sessions["n_prefix_target_pairs"] = sessions["n_groups"] - 1

cut80, cut90 = [
    int(events["timestamp"].quantile(q, interpolation="linear")) for q in QUANTILES
]
assert cut80 < cut90, "Không đủ mốc thời gian phân biệt để chia theo cấu hình này."
print("Mốc 80%:", cut80, "|", utc(cut80))
print("Mốc 90%:", cut90, "|", utc(cut90))
display(sessions.head())

## 6. Gán tập theo toàn phiên và tách phiên qua biên

| Tập | Điều kiện cho toàn phiên được giữ |
| --- | --- |
| Train | Kết thúc ≤ mốc 80% |
| Validation | Bắt đầu > mốc 80%, kết thúc ≤ mốc 90% |
| Test | Bắt đầu > mốc 90% |
| Boundary drop | Bắt đầu ≤ một mốc và kết thúc > mốc đó |

Không chia một phiên thành hai tập. Nhãn `split` được ánh xạ từ phiên xuống mọi sự kiện và nhóm. Các tệp tổng hợp vẫn giữ `boundary_drop` để truy vết; ba tệp split dùng ở bước tiếp theo loại hoàn toàn các phiên này.

Các điều kiện được kiểm tra trên ví dụ nhỏ có phiên kết thúc đúng mốc, bắt đầu đúng mốc, vượt một mốc và vượt cả hai mốc.

In [ ]:
def assign_session_splits(frame, cutoff80, cutoff90):
    require(cutoff80 < cutoff90, "Mốc 80% phải nhỏ hơn mốc 90%.")
    require(frame["start_ms"].le(frame["end_ms"]).all(), "Phiên có thời gian đầu lớn hơn thời gian cuối.")
    labels = pd.Series(pd.NA, index=frame.index, dtype="string")
    labels.loc[frame["end_ms"].le(cutoff80)] = "train"
    labels.loc[frame["start_ms"].gt(cutoff80) & frame["end_ms"].le(cutoff90)] = "validation"
    labels.loc[frame["start_ms"].gt(cutoff90)] = "test"
    boundary = pd.Series(False, index=frame.index)
    for cutoff in [cutoff80, cutoff90]:
        boundary |= frame["start_ms"].le(cutoff) & frame["end_ms"].gt(cutoff)
    labels.loc[boundary] = "boundary_drop"
    require(labels.notna().all(), "Có phiên chưa được gán split.")
    return pd.Series(pd.Categorical(labels, categories=SPLIT_NAMES), index=frame.index)


boundary_examples = pd.DataFrame({
    "start_ms": [0, 50, 100, 101, 190, 200, 201, 0],
    "end_ms": [100, 100, 101, 200, 201, 200, 230, 230],
}, index=[11, 13, 17, 19, 23, 29, 31, 37])
assert assign_session_splits(boundary_examples, 100, 200).tolist() == [
    "train", "train", "boundary_drop", "validation", "boundary_drop", "validation", "test", "boundary_drop"
]
print("Đã kiểm tra các trường hợp bằng mốc và cắt qua một/hai mốc.")

sessions["split"] = assign_session_splits(sessions, cut80, cut90)
crosses_boundary = sessions["split"].eq("boundary_drop")
session_split = sessions.set_index("session_id")["split"]
events["split"] = pd.Categorical(events["session_id"].map(session_split), categories=SPLIT_NAMES)
groups["split"] = pd.Categorical(groups["session_id"].map(session_split), categories=SPLIT_NAMES)

split_summary = sessions.groupby("split", observed=True).agg(
    sessions=("session_id", "size"),
    events=("n_events", "sum"),
    timestamp_groups=("n_groups", "sum"),
    prefix_target_pairs_before_candidate_filter=("n_prefix_target_pairs", "sum"),
)
display(split_summary)
print("Các cặp ở boundary_drop bị loại; chưa tạo query/target thực tế.")
del boundary_examples

## 7. Kiểm tra tính toàn vẹn và thứ tự thời gian

Kiểm tra không mất sự kiện, không chia một phiên sang nhiều tập, đúng khoảng thời gian của từng tập và mọi nhóm có một nhãn split. Đối chiếu số dòng/hành vi với báo cáo notebook 01 khi nguồn đầu vào đã được xác nhận. Kiểm tra nhóm chỉ thuộc một phiên và số thứ tự nhóm khớp số nhóm của phiên.

Đây là kiểm tra ranh giới thời gian của dữ liệu. Các bước sau vẫn phải bảo đảm metadata ≤ thời điểm query và tham số/vocabulary chỉ học từ train.

In [ ]:
assert len(events) == input_rows
assert events["event_id"].is_unique
assert int(sessions["n_events"].sum()) == input_rows
assert int(sessions["n_groups"].sum()) == len(groups)
assert events["split"].notna().all() and groups["split"].notna().all()
assert events.groupby("session_id", sort=False, observed=True)["split"].nunique().eq(1).all()
assert events.groupby("group_id", sort=False, observed=True)["timestamp"].nunique().eq(1).all()

train_sessions = sessions.loc[sessions["split"].eq("train")]
validation_sessions = sessions.loc[sessions["split"].eq("validation")]
test_sessions = sessions.loc[sessions["split"].eq("test")]
assert all(len(frame) for frame in [train_sessions, validation_sessions, test_sessions])
assert train_sessions["end_ms"].le(cut80).all()
assert validation_sessions["start_ms"].gt(cut80).all()
assert validation_sessions["end_ms"].le(cut90).all()
assert test_sessions["start_ms"].gt(cut90).all()
assert train_sessions["end_ms"].max() < validation_sessions["start_ms"].min()
assert validation_sessions["end_ms"].max() < test_sessions["start_ms"].min()
expected_boundary = pd.Series(False, index=sessions.index)
for cutoff in [cut80, cut90]:
    expected_boundary |= sessions["start_ms"].le(cutoff) & sessions["end_ms"].gt(cutoff)
require(expected_boundary.eq(crosses_boundary).all(), "Nhãn boundary_drop không khớp thời gian phiên.")

require(
    events.groupby("group_id", sort=False)["session_id"].nunique().eq(1).all(),
    "Một group_id chứa nhiều phiên."
)
last_group_index = groups.groupby("session_id", sort=False)["group_index"].max()
require(
    (last_group_index.reindex(sessions["session_id"]).to_numpy() == sessions["n_groups"].to_numpy()).all(),
    "Thứ tự nhóm cuối không khớp số nhóm trong phiên."
)
if upstream_verified:
    counts = {str(key): int(value) for key, value in events["event"].value_counts().items()}
    require(counts == upstream_report["clean_event_counts"], "Số sự kiện theo hành vi khác báo cáo notebook 01.")
    print("Số dòng và phân bố hành vi khớp notebook 01.")

print("Đã kiểm tra: toàn vẹn sự kiện, nhóm và ranh giới thời gian.")

## 8. Xuất bảng và đọc lại kiểm tra

| Tệp trong `data/processed/` | Nội dung |
| --- | --- |
| `events_sessionized.parquet` | Toàn bộ events, có session/group/split; gồm cả boundary_drop để truy vết |
| `sessions.parquet` | Một dòng/phiên, thời gian đầu/cuối và thống kê |
| `timestamp_groups.parquet` | Một dòng/nhóm; nối `group_id` sang bảng events để lấy đầy đủ item/hành vi |
| `splits/train.parquet` | Events của các phiên train hợp lệ |
| `splits/validation.parquet` | Events của các phiên validation hợp lệ |
| `splits/test.parquet` | Events của các phiên test hợp lệ |
| `splits/boundary_drop.parquet` | Events bị loại, lưu riêng để đối chiếu |

**Dùng ba tệp train/validation/test cho các bước tiếp theo.** Không dùng bảng tổng hợp chưa lọc làm lịch sử mô hình. Nếu đọc bảng nhóm tổng hợp, lọc `split` tương ứng hoặc nối với phiên hợp lệ.

Mỗi bảng được ghi tạm, đọc lại và so sánh toàn bộ trước khi thay thế đầu ra chính. Báo cáo ghi hai mã băm: `sha256` theo byte của tệp (truy vết phiên bản) và `content_sha256` theo nội dung đã sắp theo khóa chính (đối chiếu giữa các máy/phiên bản pyarrow). Notebook không sửa events_clean hoặc raw.

In [ ]:
require(fingerprint(SOURCE) == source_before, "Đầu vào thay đổi trong lúc chạy; cần chạy lại.")
PROCESSED.mkdir(parents=True, exist_ok=True)
SPLIT_DIR.mkdir(parents=True, exist_ok=True)
artifacts = {}


def save_verified(frame, destination, key):
    temporary = destination.with_name(destination.name + ".tmp")
    try:
        frame.to_parquet(temporary, engine="pyarrow", index=False)
        restored = pd.read_parquet(temporary, engine="pyarrow")
        pd.testing.assert_frame_equal(frame, restored, check_exact=True)
        temporary.replace(destination)
        del restored
    finally:
        temporary.unlink(missing_ok=True)
    artifacts[destination.relative_to(ROOT).as_posix()] = {
        "rows": len(frame), **fingerprint(destination),
        "content_sha256": content_sha256(frame, key), "content_key": key,
        "parquet_roundtrip_verified": True,
    }
    print(f"Đã lưu và đọc kiểm tra {destination.name}: {len(frame):,} dòng")


save_verified(events, PROCESSED / "events_sessionized.parquet", ["event_id"])
save_verified(sessions, PROCESSED / "sessions.parquet", ["session_id"])
save_verified(groups, PROCESSED / "timestamp_groups.parquet", ["group_id"])
for name in SPLIT_NAMES:
    partition = events.loc[events["split"].eq(name)].reset_index(drop=True)
    require(partition["split"].eq(name).all(), f"Tệp {name} chứa nhãn split khác.")
    save_verified(partition, SPLIT_DIR / f"{name}.parquet", ["event_id"])
del partition

## 9. Lưu cấu hình chia tập và báo cáo

Cấu hình lưu ngưỡng phiên, mốc cụ thể, cách xử lý bằng mốc và phiên qua biên. Báo cáo lưu số dòng, số nhóm, phiên, số cặp dự kiến và SHA-256 của các đầu ra đã kiểm tra. Không tính lại mốc riêng cho mỗi mô hình.

In [ ]:
split_config = {
    "source": {"path": SOURCE.relative_to(ROOT).as_posix(), **source_before},
    "timezone": "UTC",
    "inactivity_threshold_ms": INACTIVITY_MS,
    "new_session_condition": "visitor changes OR consecutive event gap > threshold",
    "timestamp_group_key": ["session_id", "timestamp"],
    "group_index_starts_at": 1,
    "quantiles": QUANTILES,
    "quantile_interpolation": "linear",
    "cutoff_conversion": "truncate to integer milliseconds",
    "cutoffs_computed_before_boundary_drop": True,
    "cutoff_ms": [cut80, cut90],
    "cutoff_utc": [utc(cut80), utc(cut90)],
    "split_rules": {
        "train": "session end <= cut80",
        "validation": "session start > cut80 AND session end <= cut90",
        "test": "session start > cut90",
        "boundary_drop": "session start <= any cutoff AND session end > that cutoff",
    },
    "exclude_boundary_drop_from_history_and_labels": True,
    "prefix_target_pairs_materialized": False,
    "boundary_examples_verified": True,
}
config_path = PROCESSED / "split_config.json"
write_json(config_path, split_config)

report = {
    "generated_at_utc": pd.Timestamp.now(tz="UTC").isoformat(),
    "python_version": sys.version.split()[0],
    "pyarrow_version": pa.__version__,
    "notebook": "notebooks/02_tao_phien_nhom_timestamp_va_chia_tap.ipynb",
    "upstream_report_verified": upstream_verified,
    "pandas_version": pd.__version__,
    "source": split_config["source"],
    "input_unchanged_during_processing": fingerprint(SOURCE) == source_before,
    "events": len(events),
    "sessions": len(sessions),
    "timestamp_groups": len(groups),
    "single_event_sessions": int(sessions["n_events"].eq(1).sum()),
    "single_timestamp_group_sessions": int(sessions["n_groups"].eq(1).sum()),
    "groups_with_multiple_events": int(groups["n_events"].gt(1).sum()),
    "groups_with_multiple_items": int(groups["n_items"].gt(1).sum()),
    "groups_with_multiple_behaviors": int(groups["n_behaviors"].gt(1).sum()),
    "splits": {
        str(name): {str(key): int(value) for key, value in row.items()}
        for name, row in split_summary.iterrows()
    },
    "split_config": {"path": config_path.relative_to(ROOT).as_posix(), **fingerprint(config_path)},
    "artifacts": artifacts,
    "scope": "Session/group/split artifacts exported; prefix-target counts only, metadata lookup and model training pending.",
}
require(report["input_unchanged_during_processing"], "Đầu vào thay đổi trước khi hoàn tất báo cáo.")
REPORT_DIR.mkdir(parents=True, exist_ok=True)
report_path = REPORT_DIR / "metrics.json"
write_json(report_path, report)
print("Cấu hình:", config_path.relative_to(ROOT).as_posix())
print("Báo cáo:", report_path.relative_to(ROOT).as_posix())
display(split_summary)

## Kết quả cần đọc và bước kế tiếp

Với bộ dữ liệu hiện tại, kết quả đã ghi nhận trước đó là 1.761.675 phiên, 2.753.015 nhóm timestamp, 47 phiên qua biên chứa 328 sự kiện. Các số trong cell được tính từ dữ liệu, không được gán cứng.

Số cặp prefix–target dự kiến trước lọc ứng viên: train 806.025, validation 92.566, test 92.468. Phiên một nhóm vẫn được lưu nhưng không tạo mẫu bước kế tiếp.

Tiếp theo chuẩn bị lịch sử `categoryid`/`available`, mã hóa item chỉ từ train và tạo query/target thực tế từ hai nhóm liên tiếp cùng phiên. Không dùng sự kiện boundary_drop làm lịch sử/nhãn hoặc thông tin sau thời điểm query. Chưa bắt đầu huấn luyện ở notebook này.